# Import packages and data loading

In [1]:
import time
import sys
import json
import pandas as pd
import random

import argparse
import os
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

# Phase 4 - Data Generation

In [2]:
def generate_instance(n, m, score_range=(0, 100), quota_factor=2, complete=True, strict=False):
    """
    n: number of applicants
    m: number of universities
    score_range: range of scores
    quota_factor: factor for determining capacity
    complete: if True, each applicant applies to all universities (complete list)
    strict: if True, for each college all applicant scores are unique (no ties), by adding a tiny epsilon. 
    """
    # generating quotas for each university
    quotas = {}
    max_quota = int(max(2, n // m) * quota_factor)
    for j in range(m):
        quotas[j] = random.randint(1, max_quota)
    
    # generating preferences and scores
    preferences = {}
    scores = {}
    for i in range(n):
        # list of universities
        if complete:
            colleges = list(range(m))
        else:
            k = max(1, int(m * 0.6))
            colleges = random.sample(range(m), k)
        random.shuffle(colleges)
        preferences[i] = colleges  # order of preference
        
        # generating scores for each university in the list
        for j in colleges:
            base_score = random.randint(score_range[0], score_range[1])
            if strict:
                score = base_score * (n + 1) + i
            else:
                score = base_score
            scores[f"{i},{j}"] = score

    return {
        "n": n,
        "m": m,
        "preferences": preferences,
        "scores": scores,
        "quotas": quotas
    }

def save_instance(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)

In [3]:
random.seed(42)

output_dir = project_root / "data" / "generated"
output_dir.mkdir(parents=True, exist_ok=True)

small_filename = "instance_small.json"
strict_small_filename = "instance_strict_small.json"
medium_filename = "instance_medium.json"
strict_medium_filename = "instance_strict_medium.json"
large_filename = "instance_large.json"
strict_large_filename = "instance_strict_large.json"

small_n, small_m = 10, 5
medium_n, medium_m = 50, 20
large_n, large_m = 1000, 20

small_score_min, small_score_max = 0, 20
medium_score_min, medium_score_max = 0, 50
large_score_min, large_score_max = 0, 50

small_quota_factor, medium_quota_factor, large_quota_factor = 1, 2, 2

In [4]:
# ---- Generate small instance ----
small_data = generate_instance(
    n=small_n,
    m=small_m,
    score_range=(small_score_min, small_score_max),
    quota_factor=small_quota_factor,
    complete=True,
    strict=False
)
small_path = output_dir / small_filename
save_instance(small_data, str(small_path))
print(f"Saved small instance to: {small_path}")

# ---- Generate medium instance ----
medium_data = generate_instance(
    n=medium_n,
    m=medium_m,
    score_range=(medium_score_min, medium_score_max),
    quota_factor=medium_quota_factor,
    complete=True,
    strict=False
)
medium_path = output_dir / medium_filename
save_instance(medium_data, str(medium_path))
print(f"Saved medium instance to: {medium_path}")

# ---- Generate large instance ----
large_data = generate_instance(
    n=large_n,
    m=large_m,
    score_range=(large_score_min, large_score_max),
    quota_factor=large_quota_factor,
    complete=True,
    strict=False
)
large_path = output_dir / large_filename
save_instance(large_data, str(large_path))
print(f"Saved large instance to: {large_path}")

# ---- Generate strict small instance ----
strict_small_data = generate_instance(
    n=small_n,
    m=small_m,
    score_range=(small_score_min, small_score_max),
    quota_factor=small_quota_factor,
    complete=True,
    strict=True
)
strict_small_path = output_dir / strict_small_filename
save_instance(strict_small_data, str(strict_small_path))
print(f"Saved strict small instance to: {strict_small_path}")

# ---- Generate strict medium instance ----
medium_strict_data = generate_instance(
    n=medium_n,
    m=medium_m,
    score_range=(medium_score_min, medium_score_max),
    quota_factor=medium_quota_factor,
    complete=True,
    strict=True
)
strict_medium_path = output_dir / strict_medium_filename
save_instance(medium_strict_data, str(strict_medium_path))
print(f"Saved strict medium instance to: {strict_medium_path}")

# ---- Generate strict large instance ----
large_strict_data = generate_instance(
    n=large_n,
    m=large_m,
    score_range=(large_score_min, large_score_max),
    quota_factor=large_quota_factor,
    complete=True,
    strict=True
)

strict_large_path = output_dir / strict_large_filename
save_instance(large_strict_data, str(strict_large_path))
print(f"Saved strict large instance to: {strict_large_path}")


Saved small instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_small.json
Saved medium instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_medium.json
Saved large instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_large.json
Saved strict small instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_strict_small.json
Saved strict medium instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_strict_medium.json
Saved strict large instance to: D:\Learning\Coding Projects\combinatorial-optimization-project\data\generated\instance_strict_large.json


# Phase 5 - Models

## Section 2 - The Gale-Shapley Model